# Growth rate $f\sigma_8(z)$ in modified gravity with EFTCAMB

Port of `Science/Peculiar_Vel/theory_plots/modified_gravity/EFTCAMB_fsigma8.ipynb`
onto `growth_review.theory.eftcamb`, which holds the model registry, the
shoot-then-solve paths and the export writer. The physics, flags, cosmology,
stability conditions and figures are that notebook's; what moved into the package
is the code, so the curves can be exported once and reused by the rest of the
review.

**This notebook needs a compiled H-EFTCAMB build** (`$EFTCAMB_PATH`, default
`/global/homes/r/ravouxco/2_Software/EFTCAMB`). It is shipped **unexecuted** for
that reason: there is no build on the laptop this was written on, and no curve
below is faked in its absence. Run it where the build lives (NERSC), then run the
export cell at the end so the other notebooks can use the results.

Everything you are likely to change lives in two places: `eftcamb.MODELS` and the
`SELECTION` list in §3.

## 1. The build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import growth_review as gr
from growth_review import theory as th
from growth_review.theory import eftcamb

gr.use_style()

print("EFTCAMB_PATH     :", eftcamb.EFTCAMB_PATH)
camb = eftcamb.camb_module()          # raises with instructions if absent
print("camb imported from:", camb.__file__)
print("camb version      :", camb.__version__)
print("EFTCAMB available :", eftcamb.available())

## 2. Baseline cosmology and stability flags

Held fixed across all models, so that every difference in the figures comes from
the gravity sector alone (Planck 2018 TT,TE,EE+lowE). `YHe` is pinned everywhere:
left automatic, CAMB's BBN predictor can return a 0-d array instead of a float,
which `set_cosmology` rejects on recent numpy.

The stability flags are the shipped examples' own: the mathematical ghost/mass
conditions off, the physical ghost and gradient conditions on -- those are the
ones that actually reject unviable parameter points.

In [ ]:
print("COSMO     :", eftcamb.COSMO)
print("YHe       :", eftcamb.YHE)
print("PK        :", eftcamb.PK_SETTINGS)
print("STABILITY :")
for k, v in eftcamb.STABILITY.items():
    print(f"    {k:28} {v}")

Z_MIN, Z_MAX, N_Z = 0.0, 2.0, 41
Z_GRID = np.linspace(Z_MIN, Z_MAX, N_Z)
REFERENCE = "GR"

## 3. Model registry

`eftcamb.MODELS` maps a name to the flags `camb.set_params` is called with. The
flag families are

| `EFTflag` | family | selector | example parameters |
|---|---|---|---|
| 0 | GR | — | — |
| 1 | pure EFT | `PureEFTmodel` | `EFTOmega0`, `EFTGamma…` |
| 2 | alternative parametrisations | `AltParEFTmodel` | `RPH…`, `OL…` |
| 3 | designer mapping | `DesignerEFTmodel=1` → $f(R)$ | `EFTB0` |
| 4 | full mapping | `FullMappingEFTmodel` | Hořava, ADE, K-mouflage, … |
| 5 | Horndeski | `Horndeski_model` | `Horndeski_param…` |

Two entries carry sentinel keys rather than flags: `_jbd_wBD` routes to the
Jordan-Brans-Dicke shoot-then-solve path (its field equations do not let the
effective cosmological-constant term be set directly). Extended Galileon
(`FullMappingEFTmodel=7`) is deliberately absent — several parameter points
background-solve in under a second and then hang at the perturbation stage.

In [ ]:
for name, spec in eftcamb.MODELS.items():
    flags = ", ".join(f"{k}={v}" for k, v in spec["params"].items()
                      if k not in eftcamb.STABILITY)
    print(f"{name:24} {flags[:96]}")

### 3.1 Designer $f(R)$ specified by $f_{R0}$

Designer $f(R)$ only *takes* $B_0$: the functional form is solved for, not chosen,
so $f_{R0}$ exists only as an output of that solve. Most of the literature quotes
$|f_{R0}|$ (the $10^{-4}, 10^{-5}, 10^{-6}$ tiers), so specifying a model that way
means inverting the direction EFTCAMB computes — `B0_for_fR0` root-finds in
$\log_{10}B_0$, and `register_fR0` wraps that into a `MODELS` entry.

$f_{R0}$ is read back off EFTCAMB's own EFT dictionary, where $\Omega(a) = f_R(a)$
for this model (`fortran/eftcamb/07f_designer_models/007p3_Designer_fR.f90`, where
`f_sub_R` is assigned to `self%EFTOmega%y`).

In [ ]:
FR0_TIERS = [-1e-4, -1e-5, -1e-6]

for target in FR0_TIERS:
    name, B0 = eftcamb.register_fR0(target)
    print(f"{name:16} f_R0 = {target:9.2e}  ->  B0 = {B0:.6e}")

In [ ]:
# Round trip: read f_R0 back through the forward evaluation (a different code
# path from the root find that produced B0), so this is a real check, not a
# tautology.
print(f"{'target f_R0':>14}  {'-> B0':>13}  {'recovered f_R0':>16}  match")
for target in FR0_TIERS:
    name = f"fR_fR0_{abs(target):.0e}"
    B0 = eftcamb.MODELS[name]["params"]["EFTB0"]
    recovered = eftcamb.fR0_from_B0(B0)
    ok = abs(recovered / target - 1.0) < 1e-6
    print(f"{target:14.3e}  {B0:13.6e}  {recovered:16.6e}  "
          f"{'OK' if ok else 'MISMATCH'}")

### 3.2 nDGP: the one model that is not an EFTCAMB run

DGP is a 5-dimensional braneworld construction, so it has no
`FullMappingEFTmodel` entry. Only its decoupling limit reduces to a covariant 4D
scalar (Chow & Khoury 2009), and the source notebook's §3.5 works through why that
embedding is unusable in practice: mapped to the Bellini-Sawicki $\alpha$ basis and
fed to EFTCAMB's RPH spline interface, it is a **ghost** on the $\dot\pi<0$ branch
(their Table 1 gives $\alpha_K$ cubic in $\alpha_M$, so its sign tracks
$\alpha_M$'s) and fails the **gradient/Laplace** condition on the other, returning
NaN $f\sigma_8$ even with both checks disabled — over $r_c = 20$–$100$ Gpc,
checked against EFTCAMB's own stability solver.

What the notebook uses instead, and what `eftcamb._ndgp_growth_ode` transcribes,
is the standard quasi-static treatment of the nDGP simulation and RSD literature
(Koyama & Maartens 2006; Schmidt 2009 eq. 2.7-2.9; Barreira, Sanchez & Schmidt
2016 eq. 7-8, the same $\beta(a)$ once $2Hr_c$ is written via $\Omega_{rc}$):

$$\mu(a) = 1 + \frac{1}{3\beta(a)}, \qquad
  \beta(a) = 1 + 2Hr_c\left(1 + \frac{\dot H}{3H^2}\right),$$

sourcing $D'' + (2 + \mathrm{d}\ln H/\mathrm{d}N)\,D' -
\tfrac{3}{2}\Omega_m(a)\,\mu(a)\,D = 0$ in $N = \ln a$, on $\Lambda$CDM's
expansion history — which the normal branch shares exactly. Scale-independent at
linear order, the opposite structural behaviour from $f(R)$'s scale-dependent
enhancement, and exact only on sub-horizon, Vainshtein-screened scales: the regime
a peculiar-velocity survey owns, but not a Boltzmann solution, and the one curve
here with no stability check of its own.

$\sigma_8$ is anchored to this notebook's own CAMB GR run at $a_i = 10^{-3}$,
where $\beta \to \infty$ so $\mu \to 1$, and propagated with
$D_{\rm nDGP}/D_{\rm GR}$ from the ODE — the shared-$A_s$ convention every other
curve here uses, so nDGP's higher $\sigma_8(0)$ is a real prediction rather than a
rescaling artefact.

### 3.3 The selection

GR first: it is the reference for the lower panel of every figure.

In [ ]:
SELECTION = [
    "GR",
    "fR_fR0_1e-04",
    "fR_fR0_1e-05",
    "fR_fR0_1e-06",
    "nDGP_H0rc1",          # not an EFTCAMB run -- see S3.2
    "nDGP_H0rc5",
    "Quintessence",
    "QuintessenceDesigner",
    "pureEFT_w0waCDM_DESI",
    "Kmouflage",
    "ScalingCubicGalileon",
    "BeyondHorndeski",
    "Horava",
    "JBD_wBD100",
]
print(len(SELECTION), "models:", ", ".join(SELECTION))

## 4. Computing $f\sigma_8(z)$

Two things `eftcamb.compute` handles that are easy to get wrong by hand:

**Redshift ordering.** CAMB re-sorts the requested redshifts internally, earliest
first, and `get_fsigma8()` returns its array in *that* order — highest $z$ first.
Zipping it against the redshift array as written silently mirrors the curve.

**Failures are informative.** EFTCAMB raises when a model violates a stability
condition it has been asked to enforce. That is a physical statement about the
parameter point, so the loop below reports the model and continues rather than
hiding it.

In [ ]:
curves, failed = {}, {}

for name in SELECTION:
    try:
        z, fs8, s8 = eftcamb.compute(name, Z_GRID)
        curves[name] = dict(z=z, fs8=fs8, s8=s8)
        print(f"{name:22} ok      fs8(0) = {fs8[0]:.4f}   sigma8(0) = {s8[0]:.4f}"
              f"   [{eftcamb.model_name(name)}]")
    except Exception as exc:
        failed[name] = exc
        print(f"{name:22} FAILED  {type(exc).__name__}: {exc}")

if failed:
    print("\nModels that did not run are usually rejected by a stability "
          "condition; relax the corresponding flag in eftcamb.STABILITY, or move "
          "the parameter point, before concluding anything physical.")

## 5. The figure

In [ ]:
# Legend and label styling shared by every figure here, so the set reads as one
# family. Transcribed from the source notebook.
LEGEND_KWARGS = dict(loc="upper left", bbox_to_anchor=(1.01, 1.02), ncol=1,
                     handlelength=2.2, labelspacing=0.5, borderaxespad=0.0)
LABEL_FONTSIZE = 20


def style_of(name):
    """(label, color, ls, lw) for a model -- the source notebook's own styling,
    kept in the package as theory.EFTCAMB_STYLE so exported curves are drawn the
    same way everywhere."""
    return th.export_style(name)


def plot_fsigma8(curves, reference=REFERENCE, ratio=True, ylim_ratio=None,
                 save=None):
    if ratio:
        fig, (ax, axr) = plt.subplots(
            2, 1, figsize=(9.4, 7.2), sharex=True,
            gridspec_kw=dict(height_ratios=[2.4, 1], hspace=0.06))
    else:
        fig, ax = plt.subplots(figsize=(9.4, 5.2))
        axr = None

    for name, c in curves.items():
        label, color, ls, lw = style_of(name)
        ax.plot(c["z"], c["fs8"], color=color, ls=ls, lw=lw, label=label, zorder=3)

    ax.set_ylabel(r"$f\sigma_8(z)$", fontsize=LABEL_FONTSIZE)
    ax.set_xlim(Z_MIN, Z_MAX)
    ax.legend(**LEGEND_KWARGS)
    ax.grid(alpha=0.18, lw=0.7)

    if ratio and reference in curves:
        ref = curves[reference]["fs8"]
        for name, c in curves.items():
            if name == reference:
                continue
            _, color, ls, lw = style_of(name)
            axr.plot(c["z"], 100.0 * (c["fs8"] / ref - 1.0), color=color, ls=ls,
                     lw=lw, zorder=3)
        axr.axhline(0.0, color="k", lw=1.6, zorder=2)
        axr.set_ylabel(r"$\Delta f\sigma_8\ [\%]$", fontsize=LABEL_FONTSIZE)
        axr.grid(alpha=0.18, lw=0.7)
        if ylim_ratio:
            axr.set_ylim(*ylim_ratio)
        axr.set_xlabel(r"redshift $z$", fontsize=LABEL_FONTSIZE)
        axr.set_xlim(Z_MIN, Z_MAX)
    else:
        ax.set_xlabel(r"redshift $z$", fontsize=LABEL_FONTSIZE)

    fig.align_ylabels()
    if save:
        fig.savefig(save, bbox_inches="tight")
        print("written:", save)
    return fig


fig = plot_fsigma8(curves, save="../figures/fsigma8_modified_gravity.pdf")
plt.show()

## 6. A scan in $B_0$

The single most useful plot for $f(R)$: how the growth enhancement builds up as
$B_0$ increases. Since $B_0$ is what surveys constrain, this is the curve that
turns a growth-rate measurement into a bound.

In [ ]:
from matplotlib import cm, colors as mcolors

B0_SCAN = np.logspace(-5, -1, 9)

scan, scan_failed = {}, []
for B0 in B0_SCAN:
    key = f"scan_B0_{B0:.3e}"
    eftcamb.MODELS[key] = dict(params=eftcamb._fr(B0), label=f"$B_0={B0:.1e}$")
    try:
        z, fs8, _ = eftcamb.compute(key, Z_GRID)
        scan[B0] = fs8
    except Exception as exc:
        scan_failed.append((B0, exc))

if scan_failed:
    print("rejected:", [f"{b:.1e}" for b, _ in scan_failed])
if not scan:
    raise SystemExit("no B0 value survived; widen B0_SCAN or relax STABILITY")

z_ref, fs8_ref, _ = eftcamb.compute(REFERENCE, Z_GRID)

norm = mcolors.LogNorm(vmin=B0_SCAN.min(), vmax=B0_SCAN.max())
cmap = cm.viridis

fig, ax = plt.subplots(figsize=(7.6, 5.2))
for B0, fs8 in scan.items():
    ax.plot(z_ref, 100 * (fs8 / fs8_ref - 1), color=cmap(norm(B0)), lw=2)
ax.axhline(0, color="k", lw=1.4)
ax.set_xlabel(r"redshift $z$")
ax.set_ylabel(r"$\Delta f\sigma_8$ relative to $\Lambda$CDM $[\%]$")
ax.set_xlim(Z_MIN, Z_MAX)
ax.grid(alpha=0.18, lw=0.7)
cb = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, pad=0.02)
cb.set_label(r"$B_0$")
fig.savefig("../figures/fsigma8_fR_B0_scan.pdf", bbox_inches="tight")
plt.show()

## 6b. Caveat: in $f(R)$ the growth rate is scale dependent

`get_fsigma8()` returns one number per redshift, built from $\sigma_8$, i.e. from
the power spectrum smoothed on $8\,h^{-1}$Mpc. That compresses away the fact that
in $f(R)$ the linear growth rate depends on **both** $z$ and $k$: the fifth force
switches on below the Compton wavelength, so small scales grow faster than large
ones. See [1906.07683](https://arxiv.org/abs/1906.07683) for a sub-per-cent
fitting function in Hu-Sawicki $f(R)$.

What this means for the curves above:

* the single $f\sigma_8(z)$ curve is an effective quantity at $8\,h^{-1}$Mpc, not
  the scale-independent growth rate of $\Lambda$CDM;
* comparing it with a measured $f\sigma_8$ is only fair if the measurement's
  effective scale and fiducial $\sigma_8$ normalisation match, and RSD and
  peculiar-velocity analyses share neither;
* the lower panel therefore under- or over-states the effect, depending on which
  scales the data actually constrain.

`eftcamb.growth_rate_kz` quantifies it: $f(k,z)$ reconstructed from the linear
power spectrum as $f = -\tfrac12\,\mathrm{d}\ln P/\mathrm{d}\ln(1+z)$. Under GR the
curves for different $k$ lie on top of each other; any spread is scale dependence.

In [ ]:
K_PROBE = [0.01, 0.05, 0.1, 0.5]         # h/Mpc
WHICH = [m for m in SELECTION if m in curves]

fig, axes = plt.subplots(1, len(WHICH), figsize=(4.1 * len(WHICH), 4.2),
                         sharey=True, squeeze=False)
for ax, name in zip(axes[0], WHICH):
    k, zz, f_kz = eftcamb.growth_rate_kz(name, np.linspace(0, 2, 21))
    for kp in K_PROBE:
        j = int(np.argmin(np.abs(k - kp)))
        ax.plot(zz, f_kz[:, j], lw=1.9, label=r"$k=%.3f\ h/$Mpc" % k[j])
    ax.set_title(style_of(name)[0], fontsize=12)
    ax.set_xlabel(r"redshift $z$")
    ax.grid(alpha=0.18, lw=0.7)
axes[0][0].set_ylabel(r"$f(k,z)$")
axes[0][-1].legend(fontsize=10)
fig.tight_layout()
fig.savefig("../figures/growth_rate_scale_dependence.pdf", bbox_inches="tight")
plt.show()

# Spread across the probed scales: how much the single fsigma8 number hides.
for name in WHICH:
    k, zz, f_kz = eftcamb.growth_rate_kz(name, np.linspace(0, 2, 21))
    js = [int(np.argmin(np.abs(k - kp))) for kp in K_PROBE]
    sel = f_kz[:, js]
    spread = (sel.max(axis=1) - sel.min(axis=1)) / sel.mean(axis=1)
    print("%-24s max spread in f over k = %.2f %%" % (name, 100 * spread.max()))

## 7. Introspecting a flag combination

`pars.EFTCAMB.param_names()` prints what a given flag combination actually
expects, which is the quickest way to check a guess before running anything.

In [ ]:
def describe(**params):
    pars = eftcamb.camb_module().set_params(**eftcamb.COSMO, YHe=eftcamb.YHE,
                                           **params)
    print("model :", pars.EFTCAMB.model_name())
    print("params:", pars.EFTCAMB.param_names())
    print("labels:", pars.EFTCAMB.param_labels())
    print("values:", pars.EFTCAMB.param_values())


describe(EFTflag=4, FullMappingEFTmodel=3, alphaU=0.2, gammaU=1, m=3.0,
         eps2_0=-0.01, gammaA=0.2, mnu=0.0, num_massive_neutrinos=0,
         **eftcamb.STABILITY)

## 8. Background evolution

$f\sigma_8(z)$ isolates a model's effect on the *growth* of structure, holding its
expansion history fixed whenever a model happens to share one. Designer $f(R)$
(`DesignerEFTmodel=1`) *builds* $\Omega(a)$ to reproduce a *prescribed* $H(z)$,
and here that prescription is $\Lambda$CDM's own — but "designer" names a
reconstruction technique, not a promise: `QuintessenceDesigner`
(`DesignerEFTmodel=2`) uses the same technique to target a genuinely different
$w_0, w_a$, so its background is *not* $\Lambda$CDM's (§9 catches this
quantitatively). Models that solve their own background from their defining
Lagrangian parameters show whatever deviation they show as a real prediction.

`eftcamb.background_ez` checks every curve against $H(z{=}0)=H_0$, true by
construction for every model here since $H_0$ is a shared input. `BeyondHorndeski`
fails that check in this build: its shooting solver converges to the right $H_0$ to
$5\times10^{-8}$, but the background `hubble_parameter()` actually tabulates
disagrees by ~44%. Its perturbation output is physically sane, so it is excluded
from the background comparison only.

In [ ]:
BACKGROUND_SELECTION = [n for n in SELECTION
                        if n in curves and n != "BeyondHorndeski"]

backgrounds = {}
for name in BACKGROUND_SELECTION:
    try:
        backgrounds[name] = eftcamb.background_ez(name, Z_GRID)
    except RuntimeError as exc:
        print(f"{name:22} excluded: {exc}")

BACKGROUND_SELECTION = [n for n in BACKGROUND_SELECTION if n in backgrounds]


def plot_background(selection, reference=REFERENCE, save=None):
    fig, (ax, axr) = plt.subplots(
        2, 1, figsize=(9.4, 7.2), sharex=True,
        gridspec_kw=dict(height_ratios=[2.4, 1], hspace=0.06))

    z_ref, E_ref = backgrounds[reference]
    for name in selection:
        label, color, ls, lw = style_of(name)
        z, Ez = backgrounds[name]
        ax.plot(z, Ez, color=color, ls=ls, lw=lw, label=label, zorder=3)

    ax.set_ylabel(r"$H(z)/H_0$", fontsize=LABEL_FONTSIZE)
    ax.set_xlim(Z_MIN, Z_MAX)
    ax.legend(**LEGEND_KWARGS)
    ax.grid(alpha=0.18, lw=0.7)

    for name in selection:
        if name == reference:
            continue
        _, color, ls, lw = style_of(name)
        z, Ez = backgrounds[name]
        axr.plot(z, 100.0 * (Ez / E_ref - 1.0), color=color, ls=ls, lw=lw, zorder=3)
    axr.axhline(0.0, color="k", lw=1.6, zorder=2)
    axr.set_ylabel(r"$\Delta H/H_{\rm GR}\ [\%]$", fontsize=LABEL_FONTSIZE)
    axr.grid(alpha=0.18, lw=0.7)
    axr.set_xlabel(r"redshift $z$", fontsize=LABEL_FONTSIZE)
    axr.set_xlim(Z_MIN, Z_MAX)

    fig.align_ylabels()
    if save:
        fig.savefig(save, bbox_inches="tight")
        print("written:", save)
    return fig


fig_bg = plot_background(BACKGROUND_SELECTION,
                         save="../figures/background_modified_gravity.pdf")
plt.show()

## 9. Background-preserving models only, at 0.1%

§8 separated models narratively into "designer-mapped, exact background by
construction" versus "solves its own background". This makes that split
quantitative, from the computed numbers rather than the model-class label — which
matters, because the label is misleading for `QuintessenceDesigner`.

A model counts as background-preserving if
$\max_z |H_{\rm model}(z)/H_{\rm GR}(z) - 1| \le 10^{-3}$ over `Z_GRID`.

`Horava` clears the cut in the source notebook ($6.65\times10^{-4}$, a real but
small deviation, well above the $\sim10^{-7}$ bisection tolerance the solvers
report) but is dropped from the showcase anyway: its $f\sigma_8$ is also
essentially $\Lambda$CDM's, so it adds a curve indistinguishable from the
reference in both panels.

In [ ]:
BACKGROUND_MOD_THRESHOLD = 1e-3          # 0.1%
BACKGROUND_MOD_THRESHOLD_1PCT = 1e-2     # 1%

_, Ez_ref = backgrounds[REFERENCE]
bg_dev = {name: float(np.max(np.abs(backgrounds[name][1] / Ez_ref - 1.0)))
          for name in BACKGROUND_SELECTION}

print(f"{'model':24} {'max|dH/H_GR|':>13}   {'0.1% cut':<12} {'1% cut':<12}")
for name in BACKGROUND_SELECTION:
    tags = ["unmodified" if bg_dev[name] <= t else "modified"
            for t in (BACKGROUND_MOD_THRESHOLD, BACKGROUND_MOD_THRESHOLD_1PCT)]
    print(f"{name:24} {bg_dev[name]:13.2e}   {tags[0]:<12} {tags[1]:<12}")

# Horava: clears both cuts, dropped by hand -- see the markdown above.
HAND_DROPPED = ["Horava"]

BACKGROUND_UNMODIFIED = [n for n in BACKGROUND_SELECTION
                         if bg_dev[n] <= BACKGROUND_MOD_THRESHOLD
                         and n not in HAND_DROPPED]
BACKGROUND_UNMODIFIED_1PCT = [n for n in BACKGROUND_SELECTION
                              if bg_dev[n] <= BACKGROUND_MOD_THRESHOLD_1PCT
                              and n not in HAND_DROPPED]

print(f"\nbackground-unmodified at 0.1% ({len(BACKGROUND_UNMODIFIED)}):",
      BACKGROUND_UNMODIFIED)
print(f"background-unmodified at 1%   ({len(BACKGROUND_UNMODIFIED_1PCT)}):",
      BACKGROUND_UNMODIFIED_1PCT)

In [ ]:
fig = plot_fsigma8({n: curves[n] for n in BACKGROUND_UNMODIFIED},
                   save="../figures/fsigma8_background_unmodified.pdf")
plt.show()

In [ ]:
fig = plot_background(BACKGROUND_UNMODIFIED,
                      save="../figures/background_unmodified.pdf")
plt.show()

## 10. The same cut at 1% — `fsigma8_background_unmodified_1pct`

§9's $10^{-3}$ cut isolates models that are background-preserving *by
construction*. Loosening it to 1% asks a different, also useful question: which
models leave $H(z)$ close enough to $\Lambda$CDM that a BAO/SN-distance analysis
at current precision would not obviously separate them, even though they are not
exact matches? In the source run that pulls in `Kmouflage` (0.63%) and
`ScalingCubicGalileon` (0.26%) on top of §9's exact-by-construction set (the three
designer $f(R)$ tiers and both nDGP entries, all at 0.00), while
`pureEFT_w0waCDM_DESI` (3.0%), `JBD_wBD100` (3.2%), `Quintessence` (5.7%) and
`QuintessenceDesigner` (6.5%) stay out — their background deviation is a real
prediction, not a threshold-sensitivity artefact.

Eight curves, then: ΛCDM, three designer $f(R)$ tiers, two nDGP benchmarks,
K-mouflage and the scaling cubic Galileon.

This is the figure `fsigma8_background_unmodified_1pct.pdf`.

In [ ]:
fig = plot_fsigma8({n: curves[n] for n in BACKGROUND_UNMODIFIED_1PCT},
                   save="../figures/fsigma8_background_unmodified_1pct.pdf")
plt.show()

In [ ]:
fig = plot_background(BACKGROUND_UNMODIFIED_1PCT,
                      save="../figures/background_background_unmodified_1pct.pdf")
plt.show()

## 11. Export, so the rest of the review can use these curves

`eftcamb.export` writes one ECSV per model — $z$, $E(z)$, $\sigma_8(z)$,
$f\sigma_8(z)$, with the flags, cosmology and stability conditions in the header.
Copy the files into `growth_review/data/theory/` on any machine and
`theory.register_exports()` turns them into ordinary models: they then appear in
`figures.fig_theory()` and in `notebooks/forecasts_vs_theory.ipynb` with the same
colours and labels as here.

Equivalently, from a shell:

```bash
growth-review-eftcamb-export --models GR Kmouflage ScalingCubicGalileon \
    Quintessence Horava JBD_wBD100 --fR0 -1e-4 -1e-5 -1e-6
```

In [ ]:
from pathlib import Path

export_dir = Path(gr.datasets.DATA_DIR) / "theory"
export_dir.mkdir(parents=True, exist_ok=True)

for name in SELECTION:
    if name not in curves:
        continue
    path = eftcamb.export(name, export_dir / f"eftcamb_{name}.ecsv", Z_GRID)
    print(f"{name:24} -> {path}")

print()
print("registered:", th.register_exports())

In [ ]:
# The exports, read back through the package's own model interface -- the same
# path every other notebook will use.
z = np.linspace(0, 2, 200)
fig, ax = plt.subplots(figsize=(9.5, 5.4))
gr.plotting.plot_theory(ax, "GR", *th.list_models(family="eftcamb"), z=z)
gr.style_axes(ax, xlim=(0, 2),
              legend_kw=dict(loc="upper left", bbox_to_anchor=(1.01, 1.02),
                             fontsize=9))
fig.tight_layout()
plt.show()

## 12. Adding a model

Append an entry to `eftcamb.MODELS` and put its name in `SELECTION`. Parameter
names come from `find_your_model/Setting and Specfying.md` and the shipped example
notebooks; §7's `describe()` prints what a flag combination expects. A model that
needs a per-call root find (like JBD) gets a sentinel key and its own path in
`growth_review/theory/eftcamb.py`, next to `_jbd_shoot`.